# 🖥️ Procesadores del Lenguaje II
## Unidad 3: Generación de Código Intermedio
### 40 Ejercicios Resueltos 
**Referencia bibliográfica:**  
Aho, A. V., Sethi, R., & Lam, M. S. (2011). *Compiladores*. Pearson Educación de México.  
[[Acceso en línea]](https://isergiobernalesgarcia.edu.pe/wp-content/uploads/2025/10/Compiladores-Alfred-V.-Aho-Monica-S.-Lam-Ravi-Sethi-Jeffrey-D.-Ullman.pdf)

---

Este cuaderno cubre los conceptos fundamentales de la **Generación de Código Intermedio**, una etapa del proceso de compilación que transforma el árbol sintáctico abstracto (AST) en una representación intermedia que es independiente de la arquitectura final.

### Temas cubiertos:
1. Código de tres direcciones
2. Cuádruplas y tripletas
3. Notación polaca inversa (RPN)
4. Generación para expresiones aritméticas
5. Generación para estructuras de control
6. Generación para arreglos y apuntadores
7. Optimización básica de código intermedio


---
## 📌 SECCIÓN 1: Código de Tres Direcciones

El **código de tres direcciones** es una representación donde cada instrucción tiene a lo más tres operandos.
La forma general es: `resultado = operando1 operador operando2`

**Tipos de instrucciones:**
- Asignación: `t1 = a + b`
- Salto incondicional: `goto L`
- Salto condicional: `if x < y goto L`
- Llamada a función: `call f, n`
- Retorno: `return x`


### ✏️ Ejercicio 1: Expresión aritmética simple
**Expresión:** `a = b + c * d`

Generar código de tres direcciones para esta expresión.

In [ ]:
# Ejercicio 1: Código de tres direcciones para a = b + c * d
# La multiplicación tiene mayor precedencia, se evalúa primero

instrucciones = [
    "t1 = c * d",      # Paso 1: evaluar c * d
    "t2 = b + t1",     # Paso 2: sumar b con el resultado
    "a  = t2"          # Paso 3: asignar a 'a'
]

print("Código de tres direcciones para: a = b + c * d")
print("-" * 40)
for i, inst in enumerate(instrucciones, 1):
    print(f"  ({i}) {inst}")

print("\n💡 Nota: Se usan variables temporales t1, t2 para almacenar resultados intermedios.")

### ✏️ Ejercicio 2: Expresión con paréntesis
**Expresión:** `x = (a + b) * (c - d)`

In [ ]:
# Ejercicio 2: (a + b) * (c - d)

instrucciones = [
    "t1 = a + b",      # Evaluar subexpresión izquierda
    "t2 = c - d",      # Evaluar subexpresión derecha
    "t3 = t1 * t2",    # Multiplicar ambas
    "x  = t3"          # Asignar resultado
]

print("Código de tres direcciones para: x = (a + b) * (c - d)")
print("-" * 45)
for i, inst in enumerate(instrucciones, 1):
    print(f"  ({i}) {inst}")

# Visualización del árbol de expresión
print("\n🌳 Árbol de expresión:")
print("       *")
print("      / \\")
print("     +   -")
print("    /\\ /\\")
print("   a b c d")

### ✏️ Ejercicio 3: Generador automático de temporales
Crear una clase que genere variables temporales y produzca código de tres direcciones.

In [ ]:
# Ejercicio 3: Clase generadora de código de tres direcciones

class GeneradorCodigoTresDirecciones:
    def __init__(self):
        self.contador_temp = 0
        self.instrucciones = []
    
    def nuevo_temporal(self):
        """Genera un nuevo nombre de variable temporal."""
        self.contador_temp += 1
        return f"t{self.contador_temp}"
    
    def emitir(self, instruccion):
        """Agrega una instrucción al código generado."""
        self.instrucciones.append(instruccion)
    
    def generar_binaria(self, op1, operador, op2):
        """Genera código para una operación binaria y retorna el temporal resultado."""
        temp = self.nuevo_temporal()
        self.emitir(f"{temp} = {op1} {operador} {op2}")
        return temp
    
    def mostrar(self):
        print("Código generado:")
        for i, inst in enumerate(self.instrucciones, 1):
            print(f"  ({i}) {inst}")

# Ejemplo: a * b + c * d
gen = GeneradorCodigoTresDirecciones()
t1 = gen.generar_binaria("a", "*", "b")  # a * b
t2 = gen.generar_binaria("c", "*", "d")  # c * d
t3 = gen.generar_binaria(t1, "+", t2)   # t1 + t2
gen.emitir(f"resultado = {t3}")

print("Expresión: resultado = a * b + c * d")
print("-" * 40)
gen.mostrar()

### ✏️ Ejercicio 4: Múltiples asignaciones en secuencia
**Programa:**
```
a = 5
b = a + 3
c = b * 2
```

In [ ]:
# Ejercicio 4: Múltiples asignaciones

gen = GeneradorCodigoTresDirecciones()

# a = 5
gen.emitir("a = 5")

# b = a + 3
t1 = gen.generar_binaria("a", "+", "3")
gen.emitir(f"b = {t1}")

# c = b * 2
t2 = gen.generar_binaria("b", "*", "2")
gen.emitir(f"c = {t2}")

print("Programa fuente:")
print("  a = 5")
print("  b = a + 3")
print("  c = b * 2")
print()
gen.mostrar()

print("\n💡 Las constantes también pueden aparecer como operandos directamente.")

### ✏️ Ejercicio 5: Expresión con negación unaria
**Expresión:** `x = -a + b`

In [ ]:
# Ejercicio 5: Negación unaria
# La negación unaria se representa como: t1 = minus a  (operador unario)

gen = GeneradorCodigoTresDirecciones()

# Negación unaria
t1 = gen.nuevo_temporal()
gen.emitir(f"{t1} = minus a")   # operación unaria

# Suma
t2 = gen.generar_binaria(t1, "+", "b")
gen.emitir(f"x = {t2}")

print("Expresión: x = -a + b")
print("-" * 35)
gen.mostrar()
print("\n💡 Los operadores unarios generan una instrucción de la forma: t = op variable")

---
## 📌 SECCIÓN 2: Cuádruplas y Tripletas

### Cuádruplas
Cada instrucción se representa como una tupla de 4 elementos: `(operador, arg1, arg2, resultado)`

### Tripletas
Igual pero sin campo de resultado; el resultado se referencia por posición: `(operador, arg1, arg2)`  
Las referencias a operaciones anteriores se hacen con `(i)` donde `i` es el número de instrucción.


### ✏️ Ejercicio 6: Representación en cuádruplas
**Expresión:** `a = b * c + b * d`

In [ ]:
# Ejercicio 6: Cuádruplas para a = b * c + b * d

# Formato: (operador, arg1, arg2, resultado)
cuadruplas = [
    ("*",  "b",  "c",  "t1"),   # t1 = b * c
    ("*",  "b",  "d",  "t2"),   # t2 = b * d
    ("+",  "t1", "t2", "t3"),   # t3 = t1 + t2
    ("=",  "t3", "",   "a"),    # a  = t3
]

print("Expresión: a = b * c + b * d")
print()
print(f"{'Pos':>4} | {'Operador':^10} | {'Arg1':^6} | {'Arg2':^6} | {'Resultado':^10}")
print("-" * 50)
for i, (op, a1, a2, res) in enumerate(cuadruplas):
    print(f"  {i:>2}  | {op:^10} | {a1:^6} | {a2:^6} | {res:^10}")

print("\n💡 Las cuádruplas son fáciles de reordenar para optimización.")
print("   El campo 'resultado' permite referencias directas por nombre.")

### ✏️ Ejercicio 7: Representación en tripletas
**Misma expresión:** `a = b * c + b * d`

In [ ]:
# Ejercicio 7: Tripletas para a = b * c + b * d
# Las referencias a resultados anteriores se hacen con (número)

tripletas = [
    (0, "*",  "b",   "c"),      # (0): b * c
    (1, "*",  "b",   "d"),      # (1): b * d
    (2, "+",  "(0)", "(1)"),    # (2): (0) + (1)   — referencia a posición 0 y 1
    (3, "=",  "a",   "(2)"),    # (3): a = (2)
]

print("Expresión: a = b * c + b * d")
print()
print(f"{'Pos':>4} | {'Operador':^10} | {'Arg1':^6} | {'Arg2':^6}")
print("-" * 38)
for pos, op, a1, a2 in tripletas:
    print(f"  ({pos})  | {op:^10} | {a1:^6} | {a2:^6}")

print("\n💡 Las tripletas ahorran espacio al no almacenar nombres de temporales.")
print("   Desventaja: difíciles de reordenar (las referencias se rompen).")

### ✏️ Ejercicio 8: Comparación cuádruplas vs tripletas
Construir ambas representaciones y compararlas.

In [ ]:
# Ejercicio 8: Comparación para  x = (a + b) - (a + c)

print("Expresión: x = (a + b) - (a + c)")
print()

# CUÁDRUPLAS
print("--- CUÁDRUPLAS ---")
cuads = [
    ("+", "a", "b", "t1"),
    ("+", "a", "c", "t2"),
    ("-", "t1", "t2", "t3"),
    ("=", "t3", "", "x"),
]
print(f"{'#':>3} {'Op':^6} {'Arg1':^5} {'Arg2':^5} {'Res':^5}")
for i, (op, a1, a2, r) in enumerate(cuads):
    print(f"  {i}  {op:^6} {a1:^5} {a2:^5} {r:^5}")

print()

# TRIPLETAS
print("--- TRIPLETAS ---")
trips = [
    (0, "+", "a", "b"),
    (1, "+", "a", "c"),
    (2, "-", "(0)", "(1)"),
    (3, "=", "x", "(2)"),
]
print(f"{'#':>3} {'Op':^6} {'Arg1':^5} {'Arg2':^5}")
for pos, op, a1, a2 in trips:
    print(f" ({pos}) {op:^6} {a1:^5} {a2:^5}")

print("\n📊 Cuádruplas: 4 campos × 4 instrucciones = 16 entradas")
print("   Tripletas:  3 campos × 4 instrucciones = 12 entradas")

### ✏️ Ejercicio 9: Tripletas indirectas
Las **tripletas indirectas** usan un arreglo de punteros a tripletas, facilitando la reordenación.

In [ ]:
# Ejercicio 9: Tripletas indirectas
# La idea: un arreglo de índices apunta a las tripletas (no a las tripletas directamente)
# Para reordenar, solo se mueve el puntero, no la tripleta

# Tabla de tripletas (el almacén)
tabla_tripletas = {
    100: ("+", "a", "b"),
    101: ("+", "a", "c"),
    102: ("-", "(100)", "(101)"),
    103: ("=", "x", "(102)"),
}

# Arreglo de ejecución (punteros)
orden_ejecucion = [100, 101, 102, 103]

print("Tabla de tripletas (almacén):")
for idx, trip in tabla_tripletas.items():
    print(f"  [{idx}] {trip}")

print("\nOrden de ejecución (punteros):")
print(" ", orden_ejecucion)

print("\nEjecución:")
for ptr in orden_ejecucion:
    op, a1, a2 = tabla_tripletas[ptr]
    print(f"  [{ptr}] {op} {a1} {a2}")

print("\n💡 Para reordenar instrucciones sin romper referencias,")
print("   solo se reordena el arreglo de punteros.")

### ✏️ Ejercicio 10: Cuádruplas para asignación de arreglo
**Instrucción:** `A[i] = B[j] + c`

In [ ]:
# Ejercicio 10: Cuádruplas para A[i] = B[j] + c
# Acceso a arreglos requiere instrucciones especiales:
#   t = A[i]   → (=[],  A, i, t)     lectura de arreglo
#   A[i] = t   → ([]=,  A, i, t)     escritura en arreglo

cuadruplas = [
    ("=[]" , "B", "j", "t1"),    # t1 = B[j]
    ("+"   , "t1", "c", "t2"),   # t2 = t1 + c
    ("[]=" , "A", "i", "t2"),    # A[i] = t2
]

print("Instrucción: A[i] = B[j] + c")
print()
print(f"{'#':>3} | {'Operador':^8} | {'Arg1':^5} | {'Arg2':^5} | {'Resultado':^10}")
print("-" * 45)
for i, (op, a1, a2, r) in enumerate(cuadruplas):
    print(f"  {i}  | {op:^8} | {a1:^5} | {a2:^5} | {r:^10}")

print("\n💡 Convenciones:")
print("   =[]  → lectura de arreglo (load)")
print("   []=  → escritura en arreglo (store)")

---
## 📌 SECCIÓN 3: Notación Polaca Inversa (RPN / Postfija)

La **Notación Polaca Inversa** coloca el operador después de sus operandos.  
Ejemplo: `a + b` → `a b +`  
Es la representación natural para una **máquina de pila**.


### ✏️ Ejercicio 11: Conversión infija → postfija (manual)
**Expresión:** `a + b * c`

In [ ]:
# Ejercicio 11: Conversión a notación postfija con el algoritmo shunting-yard

def infija_a_postfija(expresion):
    """
    Convierte una expresión infija a postfija usando el algoritmo de Dijkstra.
    Soporta: +, -, *, /, paréntesis.
    """
    precedencia = {'+': 1, '-': 1, '*': 2, '/': 2}
    salida = []
    pila = []
    tokens = expresion.split()

    for token in tokens:
        if token.isalnum():           # Operando
            salida.append(token)
        elif token == '(':             # Paréntesis abierto
            pila.append(token)
        elif token == ')':             # Paréntesis cerrado
            while pila and pila[-1] != '(':
                salida.append(pila.pop())
            pila.pop()                 # Eliminar '('
        else:                          # Operador
            while (pila and pila[-1] != '(' and
                   pila[-1] in precedencia and
                   precedencia[pila[-1]] >= precedencia[token]):
                salida.append(pila.pop())
            pila.append(token)

    while pila:
        salida.append(pila.pop())

    return ' '.join(salida)

# Pruebas
expresiones = [
    "a + b * c",
    "( a + b ) * c",
    "a + b + c",
    "( a + b ) * ( c - d )",
    "a * b + c * d",
]

print(f"{'Expresión Infija':<30} {'Postfija (RPN)':<25}")
print("-" * 55)
for expr in expresiones:
    postfija = infija_a_postfija(expr)
    print(f"{expr:<30} {postfija:<25}")

### ✏️ Ejercicio 12: Evaluación de expresión postfija con pila

In [ ]:
# Ejercicio 12: Evaluar expresión postfija usando una pila

def evaluar_postfija(expresion, valores={}):
    """
    Evalúa una expresión en notación postfija.
    valores: diccionario con valores de variables.
    """
    pila = []
    tokens = expresion.split()
    pasos = []

    for token in tokens:
        if token in '+-*/':
            b = pila.pop()
            a = pila.pop()
            if token == '+': res = a + b
            elif token == '-': res = a - b
            elif token == '*': res = a * b
            elif token == '/': res = a / b
            pila.append(res)
            pasos.append(f"  Pop {a}, {b}  →  {a} {token} {b} = {res}  →  Pila: {pila}")
        else:
            val = valores.get(token, float(token) if token.replace('.','').isdigit() else token)
            pila.append(val)
            pasos.append(f"  Push {val}  →  Pila: {pila}")

    return pila[0], pasos

# Ejemplo: (3 + 4) * 2  →  postfija: 3 4 + 2 *
expr_postfija = "3 4 + 2 *"
resultado, pasos = evaluar_postfija(expr_postfija)

print(f"Expresión postfija: {expr_postfija}")
print(f"Equivalente infija: (3 + 4) * 2")
print("\nTrazado de pila:")
for paso in pasos:
    print(paso)
print(f"\n✅ Resultado: {resultado}")

### ✏️ Ejercicio 13: Generación de código de pila desde postfija

In [ ]:
# Ejercicio 13: Generar instrucciones para máquina de pila desde expresión postfija

def generar_codigo_pila(expr_postfija):
    """
    Genera instrucciones tipo máquina de pila:
    PUSH x  → empuja x a la pila
    ADD     → saca dos, suma, empuja resultado
    SUB     → resta
    MUL     → multiplica
    DIV     → divide
    """
    ops_nombre = {'+': 'ADD', '-': 'SUB', '*': 'MUL', '/': 'DIV'}
    instrucciones = []
    tokens = expr_postfija.split()

    for token in tokens:
        if token in ops_nombre:
            instrucciones.append(ops_nombre[token])
        else:
            instrucciones.append(f"PUSH {token}")

    return instrucciones

# Ejemplo: a + b * c  →  postfija: a b c * +
postfija = "a b c * +"
codigo = generar_codigo_pila(postfija)

print(f"Expresión infija:  a + b * c")
print(f"Expresión postfija: {postfija}")
print("\nCódigo para máquina de pila:")
for i, inst in enumerate(codigo, 1):
    print(f"  {i:>2}. {inst}")

### ✏️ Ejercicio 14: De AST a código de tres direcciones
Recorrer un árbol de expresión y generar código automáticamente.

In [ ]:
# Ejercicio 14: Árbol de Sintaxis Abstracta → Código de Tres Direcciones

class Nodo:
    """Nodo del árbol de expresión."""
    def __init__(self, valor, izq=None, der=None):
        self.valor = valor
        self.izq = izq
        self.der = der

contador = [0]
codigo_generado = []

def nuevo_temp():
    contador[0] += 1
    return f"t{contador[0]}"

def generar_desde_ast(nodo):
    """Recorre el AST en postorden y genera código de tres direcciones."""
    operadores = {'+', '-', '*', '/'}
    
    if nodo.valor not in operadores:  # Es hoja (variable o constante)
        return nodo.valor
    
    # Recursión en hijos
    izq = generar_desde_ast(nodo.izq)
    der = generar_desde_ast(nodo.der)
    
    # Generar instrucción
    temp = nuevo_temp()
    codigo_generado.append(f"{temp} = {izq} {nodo.valor} {der}")
    return temp

# Construir AST para: (a + b) * (c - d)
#         *
#        / \
#       +   -
#      /\ /\
#     a b c d
ast = Nodo('*',
    Nodo('+', Nodo('a'), Nodo('b')),
    Nodo('-', Nodo('c'), Nodo('d'))
)

resultado = generar_desde_ast(ast)

print("Expresión: (a + b) * (c - d)")
print("\n🌳 AST construido manualmente.")
print("\nCódigo de tres direcciones generado:")
for i, inst in enumerate(codigo_generado, 1):
    print(f"  ({i}) {inst}")
print(f"  Resultado final en: {resultado}")

### ✏️ Ejercicio 15: Visualización del árbol de expresión

In [ ]:
# Ejercicio 15: Imprimir el árbol de expresión de forma visual

def imprimir_arbol(nodo, prefijo="", es_izquierdo=True):
    """Imprime el árbol de forma visual tipo árbol."""
    if nodo is None:
        return
    conector = "├── " if es_izquierdo else "└── "
    print(prefijo + conector + str(nodo.valor))
    
    extension = "│   " if es_izquierdo else "    "
    if nodo.izq or nodo.der:
        if nodo.izq:
            imprimir_arbol(nodo.izq, prefijo + extension, True)
        if nodo.der:
            imprimir_arbol(nodo.der, prefijo + extension, False)

# AST para: a + b * c - d
#       -
#      / \
#     +   d
#    / \
#   a   *
#      / \
#     b   c
ast2 = Nodo('-',
    Nodo('+',
        Nodo('a'),
        Nodo('*', Nodo('b'), Nodo('c'))
    ),
    Nodo('d')
)

print("Árbol de expresión para: a + b * c - d")
print("─" * 40)
print("  " + ast2.valor)
imprimir_arbol(ast2, "  ", True)

# Generar código
contador[0] = 0
codigo_generado.clear()
res = generar_desde_ast(ast2)
print("\nCódigo de tres direcciones:")
for i, inst in enumerate(codigo_generado, 1):
    print(f"  ({i}) {inst}")

---
## 📌 SECCIÓN 4: Estructuras de Control

Las estructuras de control (if, while, for) se traducen usando **saltos condicionales e incondicionales**.

```
if (cond) S1 else S2
  →  evaluar cond
     ifFalse cond goto L1
     [código S1]
     goto L2
  L1: [código S2]
  L2: ...
```


### ✏️ Ejercicio 16: Código para if-else simple
```python
if a > b:
    x = a - b
else:
    x = b - a
```

In [ ]:
# Ejercicio 16: if-else

class GeneradorControl:
    def __init__(self):
        self.instrucciones = []
        self.contador_labels = 0
        self.contador_temp = 0

    def nueva_etiqueta(self):
        self.contador_labels += 1
        return f"L{self.contador_labels}"

    def nuevo_temp(self):
        self.contador_temp += 1
        return f"t{self.contador_temp}"

    def emitir(self, inst):
        self.instrucciones.append(inst)

    def mostrar(self):
        for inst in self.instrucciones:
            print(f"  {inst}")

gen = GeneradorControl()
L_else = gen.nueva_etiqueta()   # L1: inicio del else
L_fin  = gen.nueva_etiqueta()   # L2: fin del if-else

# Condición
t1 = gen.nuevo_temp()
gen.emitir(f"{t1} = a > b")
gen.emitir(f"ifFalse {t1} goto {L_else}")

# Rama then
t2 = gen.nuevo_temp()
gen.emitir(f"{t2} = a - b")
gen.emitir(f"x = {t2}")
gen.emitir(f"goto {L_fin}")

# Rama else
gen.emitir(f"{L_else}:")
t3 = gen.nuevo_temp()
gen.emitir(f"{t3} = b - a")
gen.emitir(f"x = {t3}")

gen.emitir(f"{L_fin}:")

print("Programa fuente:")
print("  if a > b:")
print("      x = a - b")
print("  else:")
print("      x = b - a")
print("\nCódigo de tres direcciones:")
gen.mostrar()

### ✏️ Ejercicio 17: Código para ciclo while

In [ ]:
# Ejercicio 17: while (i < n): suma = suma + i; i = i + 1

gen = GeneradorControl()
L_inicio = gen.nueva_etiqueta()  # Inicio del ciclo (volver aquí)
L_fin    = gen.nueva_etiqueta()  # Salir del ciclo

gen.emitir(f"{L_inicio}:")

# Condición del while
t1 = gen.nuevo_temp()
gen.emitir(f"{t1} = i < n")
gen.emitir(f"ifFalse {t1} goto {L_fin}")

# Cuerpo del while
t2 = gen.nuevo_temp()
gen.emitir(f"{t2} = suma + i")
gen.emitir(f"suma = {t2}")

t3 = gen.nuevo_temp()
gen.emitir(f"{t3} = i + 1")
gen.emitir(f"i = {t3}")

gen.emitir(f"goto {L_inicio}")
gen.emitir(f"{L_fin}:")

print("Programa fuente:")
print("  while i < n:")
print("      suma = suma + i")
print("      i = i + 1")
print("\nCódigo de tres direcciones:")
gen.mostrar()

### ✏️ Ejercicio 18: Código para ciclo for

In [ ]:
# Ejercicio 18: for i = 0 to n-1: suma = suma + A[i]
# El for se transforma en: inicialización + while

gen = GeneradorControl()
L_inicio = gen.nueva_etiqueta()
L_fin    = gen.nueva_etiqueta()

# Inicialización
gen.emitir("i = 0")

gen.emitir(f"{L_inicio}:")

# Condición
t1 = gen.nuevo_temp()
gen.emitir(f"{t1} = i < n")
gen.emitir(f"ifFalse {t1} goto {L_fin}")

# Cuerpo
t2 = gen.nuevo_temp()
gen.emitir(f"{t2} = A[i]")      # Acceso al arreglo
t3 = gen.nuevo_temp()
gen.emitir(f"{t3} = suma + {t2}")
gen.emitir(f"suma = {t3}")

# Incremento
t4 = gen.nuevo_temp()
gen.emitir(f"{t4} = i + 1")
gen.emitir(f"i = {t4}")

gen.emitir(f"goto {L_inicio}")
gen.emitir(f"{L_fin}:")

print("Programa fuente:")
print("  for i = 0 to n-1:")
print("      suma = suma + A[i]")
print("\nCódigo de tres direcciones:")
gen.mostrar()

### ✏️ Ejercicio 19: If sin else (if simple)

In [ ]:
# Ejercicio 19: if sin else
# if (x > 0): y = x * 2

gen = GeneradorControl()
L_fin = gen.nueva_etiqueta()

# Condición
t1 = gen.nuevo_temp()
gen.emitir(f"{t1} = x > 0")
gen.emitir(f"ifFalse {t1} goto {L_fin}")  # Si falso, saltar el cuerpo

# Cuerpo
t2 = gen.nuevo_temp()
gen.emitir(f"{t2} = x * 2")
gen.emitir(f"y = {t2}")

gen.emitir(f"{L_fin}:")

print("Programa fuente:")
print("  if x > 0:")
print("      y = x * 2")
print("\nCódigo de tres direcciones:")
gen.mostrar()
print("\n💡 Sin else, solo se necesita 1 etiqueta al final del bloque.")

### ✏️ Ejercicio 20: If anidado

In [ ]:
# Ejercicio 20: if anidado
# if a > 0:
#     if b > 0:
#         c = a + b
#     else:
#         c = a - b
# else:
#     c = 0

gen = GeneradorControl()
L_else_externo = gen.nueva_etiqueta()  # L1
L_else_interno = gen.nueva_etiqueta()  # L2
L_fin_interno  = gen.nueva_etiqueta()  # L3
L_fin          = gen.nueva_etiqueta()  # L4

# if a > 0
t1 = gen.nuevo_temp()
gen.emitir(f"{t1} = a > 0")
gen.emitir(f"ifFalse {t1} goto {L_else_externo}")

  # if b > 0
t2 = gen.nuevo_temp()
gen.emitir(f"  {t2} = b > 0")
gen.emitir(f"  ifFalse {t2} goto {L_else_interno}")
  # c = a + b
t3 = gen.nuevo_temp()
gen.emitir(f"  {t3} = a + b")
gen.emitir(f"  c = {t3}")
gen.emitir(f"  goto {L_fin_interno}")
  # else: c = a - b
gen.emitir(f"  {L_else_interno}:")
t4 = gen.nuevo_temp()
gen.emitir(f"  {t4} = a - b")
gen.emitir(f"  c = {t4}")
gen.emitir(f"  {L_fin_interno}:")
gen.emitir(f"goto {L_fin}")

# else externo: c = 0
gen.emitir(f"{L_else_externo}:")
gen.emitir(f"c = 0")

gen.emitir(f"{L_fin}:")

print("Código de tres direcciones para if anidado:")
gen.mostrar()

---
## 📌 SECCIÓN 5: Funciones y Llamadas a Procedimientos

La llamada a funciones implica:
1. Pasar parámetros (`param x`)
2. Llamar la función (`call f, n`)
3. Obtener el valor de retorno


### ✏️ Ejercicio 21: Llamada a función con parámetros

In [ ]:
# Ejercicio 21: Llamada a función y= max(a, b)

instrucciones = [
    "param a",         # Pasar argumento 1
    "param b",         # Pasar argumento 2
    "call max, 2",     # Llamar a max con 2 parámetros
    "y = retval",      # Obtener valor de retorno
]

print("Programa fuente: y = max(a, b)")
print("\nCódigo de tres direcciones:")
for i, inst in enumerate(instrucciones, 1):
    print(f"  ({i}) {inst}")

print("\n💡 Convenciones:")
print("   param x     → empuja x como argumento")
print("   call f, n   → llama a f con n parámetros")
print("   retval      → registro especial con el valor de retorno")

### ✏️ Ejercicio 22: Definición de función con código intermedio

In [ ]:
# Ejercicio 22: Definición de función max(a, b)
# int max(int a, int b) { if (a > b) return a; else return b; }

print("Función fuente:")
print("  int max(int a, int b) {")
print("      if (a > b) return a;")
print("      else return b;")
print("  }")
print()

gen = GeneradorControl()
L_else = gen.nueva_etiqueta()
L_fin  = gen.nueva_etiqueta()

gen.emitir("func max:")

t1 = gen.nuevo_temp()
gen.emitir(f"  {t1} = a > b")
gen.emitir(f"  ifFalse {t1} goto {L_else}")
gen.emitir(f"  return a")
gen.emitir(f"  goto {L_fin}")
gen.emitir(f"  {L_else}:")
gen.emitir(f"  return b")
gen.emitir(f"  {L_fin}:")
gen.emitir("endfunc")

print("Código de tres direcciones:")
gen.mostrar()

### ✏️ Ejercicio 23: Función recursiva — factorial

In [ ]:
# Ejercicio 23: int factorial(int n) { if n<=1 return 1; return n * factorial(n-1); }

print("Función fuente:")
print("  int factorial(int n) {")
print("      if (n <= 1) return 1;")
print("      return n * factorial(n-1);")
print("  }")
print()

gen = GeneradorControl()
L_base = gen.nueva_etiqueta()
L_rec  = gen.nueva_etiqueta()

gen.emitir("func factorial:")

# if n <= 1 return 1
t1 = gen.nuevo_temp()
gen.emitir(f"  {t1} = n <= 1")
gen.emitir(f"  ifFalse {t1} goto {L_rec}")
gen.emitir(f"  return 1")

# return n * factorial(n-1)
gen.emitir(f"  {L_rec}:")
t2 = gen.nuevo_temp()
gen.emitir(f"  {t2} = n - 1")
gen.emitir(f"  param {t2}")            # Argumento para llamada recursiva
gen.emitir(f"  call factorial, 1")
t3 = gen.nuevo_temp()
gen.emitir(f"  {t3} = retval")
t4 = gen.nuevo_temp()
gen.emitir(f"  {t4} = n * {t3}")
gen.emitir(f"  return {t4}")

gen.emitir("endfunc")

print("Código de tres direcciones:")
gen.mostrar()

---
## 📌 SECCIÓN 6: Código Intermedio para Arreglos

Para acceder a un arreglo A[i], se debe calcular la **dirección efectiva**:
- Si los elementos tienen tamaño `w`: dirección = base(A) + i * w
- Para arreglos 2D A[i][j]: dirección = base + (i * n + j) * w


### ✏️ Ejercicio 24: Acceso a arreglo unidimensional

In [ ]:
# Ejercicio 24: Acceso a A[i] donde cada elemento ocupa w=4 bytes
# x = A[i]

print("Instrucción fuente: x = A[i]  (elementos de tamaño w=4)")
print("\nCódigo de tres direcciones (con cálculo de offset):")

instrucciones = [
    "t1 = i * 4",          # Calcular offset (índice × tamaño)
    "t2 = base_A + t1",    # Calcular dirección efectiva
    "x  = *t2",            # Desreferenciar puntero (leer memoria)
]

for i, inst in enumerate(instrucciones, 1):
    print(f"  ({i}) {inst}")

print("\n--- Forma simplificada (notación de arreglo):")
instrucciones2 = ["x = A[i]"]
for i, inst in enumerate(instrucciones2, 1):
    print(f"  ({i}) {inst}")

print("\n💡 La forma simplificada es común en representaciones de alto nivel.")
print("   La forma expandida se usa cuando se genera código de máquina real.")

### ✏️ Ejercicio 25: Acceso a arreglo bidimensional

In [ ]:
# Ejercicio 25: Acceso a A[i][j] en arreglo de n=10 columnas, w=4 bytes
# x = A[i][j]

n = 10   # número de columnas
w = 4    # tamaño de cada elemento en bytes

print(f"Instrucción fuente: x = A[i][j]  (arreglo {n} cols, elementos {w} bytes)")
print(f"Fórmula: offset = (i * {n} + j) * {w}")
print("\nCódigo de tres direcciones:")

instrucciones = [
    f"t1 = i * {n}",         # i * numero_columnas
    f"t2 = t1 + j",          # posición lineal
    f"t3 = t2 * {w}",        # offset en bytes
    f"t4 = base_A + t3",     # dirección efectiva
    f"x  = *t4",             # leer valor
]

for i, inst in enumerate(instrucciones, 1):
    print(f"  ({i}) {inst}")

### ✏️ Ejercicio 26: Copiar un arreglo (loop + acceso)

In [ ]:
# Ejercicio 26: Copiar arreglo B en A: for i=0 to n-1: A[i] = B[i]

gen = GeneradorControl()
L_inicio = gen.nueva_etiqueta()
L_fin    = gen.nueva_etiqueta()

gen.emitir("i = 0")
gen.emitir(f"{L_inicio}:")

t1 = gen.nuevo_temp()
gen.emitir(f"{t1} = i < n")
gen.emitir(f"ifFalse {t1} goto {L_fin}")

# A[i] = B[i]
t2 = gen.nuevo_temp()
gen.emitir(f"{t2} = B[i]")     # Leer B[i]
gen.emitir(f"A[i] = {t2}")     # Escribir en A[i]

t3 = gen.nuevo_temp()
gen.emitir(f"{t3} = i + 1")
gen.emitir(f"i = {t3}")
gen.emitir(f"goto {L_inicio}")
gen.emitir(f"{L_fin}:")

print("Programa: copiar B en A")
print("  for i = 0 to n-1:")
print("      A[i] = B[i]")
print("\nCódigo de tres direcciones:")
gen.mostrar()

---
## 📌 SECCIÓN 7: Optimización Básica de Código Intermedio

Técnicas de optimización básica:
1. **Eliminación de subexpresiones comunes** — si `t1 = a + b` aparece dos veces, reutilizar `t1`
2. **Propagación de constantes** — reemplazar variables conocidas por su valor
3. **Eliminación de código muerto** — quitar instrucciones cuyos resultados nunca se usan
4. **Reducción de fuerza** — reemplazar `x*2` por `x+x`


### ✏️ Ejercicio 27: Eliminación de subexpresiones comunes

In [ ]:
# Ejercicio 27: Eliminación de subexpresiones comunes (CSE - Common Subexpression Elimination)

# Código original con redundancia
original = [
    "t1 = a + b",
    "t2 = a + b",    # ← subexpresión repetida
    "t3 = t1 * c",
    "t4 = t2 * d",
]

# Código optimizado
optimizado = [
    "t1 = a + b",
    "# t2 = a + b  (eliminada, se usa t1)",
    "t3 = t1 * c",
    "t4 = t1 * d",   # ← t2 reemplazado por t1
]

print("Código original:")
for i, inst in enumerate(original, 1):
    print(f"  ({i}) {inst}")

print("\nCódigo optimizado (CSE aplicada):")
for i, inst in enumerate(optimizado, 1):
    print(f"  ({i}) {inst}")

print("\n✅ Se eliminó 1 instrucción y se ahorró 1 operación suma.")

### ✏️ Ejercicio 28: Propagación de constantes

In [ ]:
# Ejercicio 28: Propagación de constantes
# Si sabemos que x = 5, podemos sustituir x por 5 en el código

def propagacion_constantes(instrucciones):
    """Aplica propagación de constantes simple."""
    constantes = {}
    resultado = []
    
    for inst in instrucciones:
        partes = inst.split(" = ")
        if len(partes) == 2:
            var, expr = partes[0].strip(), partes[1].strip()
            
            # Sustituir constantes conocidas en la expresión
            expr_original = expr
            for cst_var, cst_val in constantes.items():
                expr = expr.replace(cst_var, str(cst_val))
            
            # Si la expresión es solo un número, guardarla
            try:
                valor = eval(expr)  # Evaluar expresión si es posible
                constantes[var] = valor
                resultado.append(f"{var} = {valor}  " + (f"  # era: {var} = {expr_original}" if expr != expr_original else ""))
            except:
                resultado.append(f"{var} = {expr}")
                if var in constantes:
                    del constantes[var]  # Ya no sabemos el valor
        else:
            resultado.append(inst)
    
    return resultado, constantes

original = [
    "x = 5",
    "y = x + 3",
    "z = y * 2",
    "w = z - x",
]

optimizado, consts = propagacion_constantes(original)

print("Código original:")
for inst in original:
    print(f"  {inst}")

print("\nCódigo con propagación de constantes:")
for inst in optimizado:
    print(f"  {inst}")

print(f"\nConstantes detectadas: {consts}")

### ✏️ Ejercicio 29: Reducción de fuerza (Strength Reduction)

In [ ]:
# Ejercicio 29: Reducción de fuerza
# Reemplazar operaciones costosas por equivalentes más baratas

print("Reducción de fuerza — ejemplos:")
print()

ejemplos = [
    ("t1 = x * 2",    "t1 = x + x",      "Mult por 2 → Suma"),
    ("t2 = x * 4",    "t2 = x << 2",     "Mult por pot. 2 → Shift izq."),
    ("t3 = x / 2",    "t3 = x >> 1",     "Div por 2 → Shift der."),
    ("t4 = x ** 2",   "t4 = x * x",      "Potencia 2 → Multiplicación"),
    ("t5 = x % 8",    "t5 = x & 7",      "Módulo pot. 2 → AND bit"),
]

print(f"{'Original':<20} {'Optimizado':<20} {'Razón':<30}")
print("-" * 70)
for orig, opt, razon in ejemplos:
    print(f"{orig:<20} {opt:<20} {razon:<30}")

print("\n💡 Los shifts son instrucciones de 1 ciclo; las multiplicaciones suelen ser 3-5 ciclos.")

### ✏️ Ejercicio 30: Eliminación de código muerto

In [ ]:
# Ejercicio 30: Eliminación de código muerto
# Código muerto: instrucciones cuyos resultados nunca se usan

instrucciones = [
    ("t1", "t1 = a + b"),   # t1 se usa en t3
    ("t2", "t2 = c * d"),   # t2 NO se usa en ningún lado → MUERTO
    ("t3", "t3 = t1 + e"),  # t3 se usa en resultado
    ("t4", "t4 = f - g"),   # t4 NO se usa → MUERTO
    ("resultado", "resultado = t3"),  # variable final
]

# Determinar qué temporales se usan
usados = set()
for var, inst in instrucciones:
    # Analizar lado derecho de la asignación
    lado_der = inst.split(" = ")[1] if " = " in inst else ""
    for v, _ in instrucciones:
        if v in lado_der:
            usados.add(v)

# Siempre mantener la variable final
usados.add("resultado")

print("Código original:")
for var, inst in instrucciones:
    estado = "✅ útil" if var in usados else "❌ muerto"
    print(f"  {inst:<30} ← {estado}")

print("\nCódigo después de eliminar código muerto:")
for var, inst in instrucciones:
    if var in usados:
        print(f"  {inst}")

print(f"\n✅ Instrucciones eliminadas: {len(instrucciones) - len(usados)}")

---
## 📌 SECCIÓN 8: Ejercicios Integradores
Ejercicios que combinan varios conceptos de la unidad.

### ✏️ Ejercicio 31: Compilador miniatura — de fuente a código intermedio

In [ ]:
# Ejercicio 31: Mini compilador — analiza tokens y genera código intermedio
# Soporta: asignaciones simples como  x = a + b  o  x = a * b + c

import re

class MiniCompilador:
    def __init__(self):
        self.codigo = []
        self.temp_count = 0
    
    def nuevo_temp(self):
        self.temp_count += 1
        return f"t{self.temp_count}"
    
    def compilar_asignacion(self, linea):
        """Procesa: var = expr"""
        var, expr = [x.strip() for x in linea.split("=", 1)]
        resultado = self.compilar_expr(expr.strip())
        self.codigo.append(f"{var} = {resultado}")
    
    def compilar_expr(self, expr):
        """Genera código para expresión con + y * (sin paréntesis)."""
        # Dividir por +
        if '+' in expr:
            partes = expr.split('+', 1)
            izq = self.compilar_expr(partes[0].strip())
            der = self.compilar_expr(partes[1].strip())
            temp = self.nuevo_temp()
            self.codigo.append(f"{temp} = {izq} + {der}")
            return temp
        # Dividir por *
        elif '*' in expr:
            partes = expr.split('*', 1)
            izq = self.compilar_expr(partes[0].strip())
            der = self.compilar_expr(partes[1].strip())
            temp = self.nuevo_temp()
            self.codigo.append(f"{temp} = {izq} * {der}")
            return temp
        else:
            return expr.strip()
    
    def compilar(self, programa):
        for linea in programa.strip().split('\n'):
            linea = linea.strip()
            if linea and '=' in linea:
                self.compilar_asignacion(linea)
        return self.codigo

programa = """
a = 2
b = a * 3
c = b + a * 4
"""

comp = MiniCompilador()
codigo = comp.compilar(programa)

print("Programa fuente:")
for l in programa.strip().split('\n'):
    print(f"  {l.strip()}")
print("\nCódigo intermedio generado:")
for i, inst in enumerate(codigo, 1):
    print(f"  ({i}) {inst}")

### ✏️ Ejercicio 32: Contar variables temporales necesarias

In [ ]:
# Ejercicio 32: Calcular el número de temporales necesarios para una expresión
# Usando la regla de Ershov (número de registros necesarios)

class NodoEj32:
    def __init__(self, valor, izq=None, der=None):
        self.valor = valor
        self.izq = izq
        self.der = der

def numero_ershov(nodo):
    """
    Calcula el número de Ershov (mínimo de registros/temporales).
    - Hoja: necesita 1
    - Si ambos hijos tienen el mismo número n: necesita n+1
    - Si izq > der: necesita izq
    - Si der > izq: necesita der (pero genera código diferente)
    """
    if nodo.izq is None and nodo.der is None:
        return 1
    
    n_izq = numero_ershov(nodo.izq)
    n_der = numero_ershov(nodo.der)
    
    if n_izq == n_der:
        return n_izq + 1
    else:
        return max(n_izq, n_der)

# Expresión: (a + b) * (c + d)
ast = NodoEj32('*',
    NodoEj32('+', NodoEj32('a'), NodoEj32('b')),
    NodoEj32('+', NodoEj32('c'), NodoEj32('d'))
)

n = numero_ershov(ast)
print("Expresión: (a + b) * (c + d)")
print(f"Número de Ershov (temporales mínimos): {n}")

# Otro ejemplo: a + b + c
ast2 = NodoEj32('+',
    NodoEj32('+', NodoEj32('a'), NodoEj32('b')),
    NodoEj32('c')
)
n2 = numero_ershov(ast2)
print(f"\nExpresión: a + b + c")
print(f"Número de Ershov: {n2}")

### ✏️ Ejercicio 33: Bloques básicos — identificar límites

In [ ]:
# Ejercicio 33: Identificar bloques básicos en código de tres direcciones
# Un bloque básico es una secuencia de instrucciones sin saltos internos

def identificar_bloques(instrucciones):
    """
    Reglas para inicio de bloque:
    1. La primera instrucción
    2. Cualquier instrucción que sea destino de un salto
    3. Instrucción inmediatamente después de un salto
    """
    n = len(instrucciones)
    liders = set()
    liders.add(0)  # Regla 1: primera instrucción
    
    for i, inst in enumerate(instrucciones):
        if 'goto' in inst:     # Instrucción de salto
            liders.add(i + 1)  # Regla 3: siguiente instrucción
            # Regla 2: buscar etiqueta destino
            destino = inst.split('goto')[1].strip()
            for j, inst2 in enumerate(instrucciones):
                if inst2.strip().startswith(destino + ':'):
                    liders.add(j)

    # Construir bloques
    liders_ord = sorted(liders)
    bloques = []
    for k, inicio in enumerate(liders_ord):
        fin = liders_ord[k + 1] if k + 1 < len(liders_ord) else n
        bloques.append((inicio, instrucciones[inicio:fin]))
    
    return bloques

codigo = [
    "t1 = a > b",           # 0
    "ifFalse t1 goto L1",   # 1
    "x = a - b",            # 2
    "goto L2",              # 3
    "L1: x = b - a",        # 4
    "L2: resultado = x",    # 5
]

bloques = identificar_bloques(codigo)

print("Código de tres direcciones:")
for i, inst in enumerate(codigo):
    print(f"  [{i}] {inst}")

print("\nBloques básicos identificados:")
for num, (inicio, bloque) in enumerate(bloques, 1):
    print(f"  Bloque {num} (desde instrucción {inicio}):")
    for inst in bloque:
        print(f"    {inst}")

### ✏️ Ejercicio 34: Grafo de flujo de control (CFG)

In [ ]:
# Ejercicio 34: Construir y visualizar un grafo de flujo de control simple

class GrafoFlujo:
    def __init__(self):
        self.nodos = {}       # id_bloque → instrucciones
        self.aristas = []     # (bloque_origen, bloque_destino)

    def agregar_bloque(self, id_bloque, instrucciones):
        self.nodos[id_bloque] = instrucciones

    def agregar_arista(self, origen, destino):
        self.aristas.append((origen, destino))

    def mostrar(self):
        print("Nodos (Bloques Básicos):")
        for bid, insts in self.nodos.items():
            print(f"  [B{bid}]")
            for inst in insts:
                print(f"       {inst}")
        print("\nAristas (flujo entre bloques):")
        for origen, destino in self.aristas:
            print(f"  B{origen} → B{destino}")

# CFG para: if a > b then x = a else x = b
cfg = GrafoFlujo()
cfg.agregar_bloque(1, ["t1 = a > b", "ifFalse t1 goto B3"])
cfg.agregar_bloque(2, ["x = a", "goto B4"])
cfg.agregar_bloque(3, ["x = b"])
cfg.agregar_bloque(4, ["resultado = x"])

cfg.agregar_arista(1, 2)  # si condición verdadera
cfg.agregar_arista(1, 3)  # si condición falsa
cfg.agregar_arista(2, 4)  # goto fin
cfg.agregar_arista(3, 4)  # flujo normal

cfg.mostrar()

print("\n💡 El CFG es la base para muchas optimizaciones (análisis de vida, eliminación de bucles, etc.)")

### ✏️ Ejercicio 35: Código intermedio para expresiones booleanas

In [ ]:
# Ejercicio 35: Código para expresión booleana compuesta: if (a > b AND c < d)
# Evaluación en cortocircuito (short-circuit)

gen = GeneradorControl()
L_cuerpo  = gen.nueva_etiqueta()  # Si ambas condiciones son verdaderas
L_false   = gen.nueva_etiqueta()  # Si alguna es falsa
L_fin     = gen.nueva_etiqueta()  # Fin del if

print("Programa: if (a > b AND c < d): x = 1 else: x = 0")
print("\n--- Sin cortocircuito (evalúa todo):")
insts_sin = [
    "t1 = a > b",
    "t2 = c < d",
    "t3 = t1 AND t2",
    f"ifFalse t3 goto L_false",
    "x = 1",
    f"goto L_fin",
    "L_false: x = 0",
    "L_fin:"
]
for inst in insts_sin:
    print(f"  {inst}")

print("\n--- Con cortocircuito (AND: si primero es falso, no evaluar segundo):")
insts_cc = [
    "t1 = a > b",
    "ifFalse t1 goto L_false",   # Si a>b es falso, ya saltar
    "t2 = c < d",
    "ifFalse t2 goto L_false",   # Si c<d es falso, saltar
    "x = 1",
    "goto L_fin",
    "L_false: x = 0",
    "L_fin:"
]
for inst in insts_cc:
    print(f"  {inst}")

print("\n💡 El cortocircuito es más eficiente y necesario para evitar errores (e.g., p != null AND p.val > 0)")

### ✏️ Ejercicio 36: Generación completa para programa con función y bucle

In [ ]:
# Ejercicio 36: Programa completo — sumatoria con función
# int suma(int n) { s = 0; for i=1 to n: s = s + i; return s; }
# main: resultado = suma(10)

gen = GeneradorControl()
L_loop = gen.nueva_etiqueta()
L_fin_loop = gen.nueva_etiqueta()

# --- Definición de función suma(n) ---
gen.emitir("func suma:")
gen.emitir("  s = 0")
gen.emitir("  i = 1")
gen.emitir(f"  {L_loop}:")

t1 = gen.nuevo_temp()
gen.emitir(f"  {t1} = i <= n")
gen.emitir(f"  ifFalse {t1} goto {L_fin_loop}")

t2 = gen.nuevo_temp()
gen.emitir(f"  {t2} = s + i")
gen.emitir(f"  s = {t2}")

t3 = gen.nuevo_temp()
gen.emitir(f"  {t3} = i + 1")
gen.emitir(f"  i = {t3}")
gen.emitir(f"  goto {L_loop}")

gen.emitir(f"  {L_fin_loop}:")
gen.emitir("  return s")
gen.emitir("endfunc")
gen.emitir("")

# --- Main ---
gen.emitir("main:")
gen.emitir("  param 10")
gen.emitir("  call suma, 1")
t4 = gen.nuevo_temp()
gen.emitir(f"  {t4} = retval")
gen.emitir(f"  resultado = {t4}")

print("Programa fuente:")
print("  int suma(int n) { s=0; for i=1..n: s=s+i; return s; }")
print("  main: resultado = suma(10)")
print("\nCódigo intermedio generado:")
gen.mostrar()

### ✏️ Ejercicio 37: Tabla de símbolos básica

In [ ]:
# Ejercicio 37: Tabla de símbolos — componente esencial del compilador

class TablaSimbolos:
    def __init__(self):
        self.tabla = {}
        self.contador_offset = 0
    
    def insertar(self, nombre, tipo, ambito="global"):
        tamanios = {"int": 4, "float": 8, "char": 1, "bool": 1}
        tam = tamanios.get(tipo, 4)
        self.tabla[nombre] = {
            "tipo": tipo,
            "ambito": ambito,
            "offset": self.contador_offset,
            "tamaño": tam
        }
        self.contador_offset += tam
    
    def buscar(self, nombre):
        return self.tabla.get(nombre, None)
    
    def mostrar(self):
        print(f"{'Nombre':<12} {'Tipo':<8} {'Ámbito':<10} {'Offset':<8} {'Tamaño':<6}")
        print("-" * 50)
        for nombre, info in self.tabla.items():
            print(f"{nombre:<12} {info['tipo']:<8} {info['ambito']:<10} {info['offset']:<8} {info['tamaño']:<6}")

# Ejemplo: programa con declaración de variables
ts = TablaSimbolos()
ts.insertar("a", "int")
ts.insertar("b", "int")
ts.insertar("c", "float")
ts.insertar("flag", "bool")
ts.insertar("ch", "char")
ts.insertar("suma", "float")

print("Tabla de Símbolos:")
ts.mostrar()

print(f"\nBuscar 'c': {ts.buscar('c')}")
print(f"Buscar 'z': {ts.buscar('z')}  ← variable no declarada")

### ✏️ Ejercicio 38: Verificador de tipos básico

In [ ]:
# Ejercicio 38: Verificación de tipos en expresiones
# Se verifica que los tipos sean compatibles antes de generar código

tipos_operandos = {
    "a": "int", "b": "int", "c": "float", "d": "float", "flag": "bool"
}

def tipo_resultado(t1, op, t2):
    """Determina el tipo resultado de una operación."""
    # Reglas de promoción de tipos
    numericos = {"int", "float"}
    if t1 in numericos and t2 in numericos:
        if t1 == "float" or t2 == "float":
            return "float"  # float domina
        return "int"
    if op in {"+", "-", "*", "/"}:
        return f"ERROR: no se puede {op} con {t1} y {t2}"
    return "bool"

expresiones = [
    ("a", "+", "b"),       # int + int
    ("a", "+", "c"),       # int + float
    ("c", "*", "d"),       # float * float
    ("a", "+", "flag"),    # int + bool → error
    ("c", "-", "a"),       # float - int
]

print(f"{'Expresión':<20} {'Tipo Resultado':<15} {'Válida':<6}")
print("-" * 45)
for op1, oper, op2 in expresiones:
    t1 = tipos_operandos[op1]
    t2 = tipos_operandos[op2]
    res = tipo_resultado(t1, oper, t2)
    valida = "✅" if not res.startswith("ERROR") else "❌"
    expr = f"{op1}({t1}) {oper} {op2}({t2})"
    print(f"{expr:<25} {res:<20} {valida}")

### ✏️ Ejercicio 39: Traducción completa de expresión a múltiples formatos

In [ ]:
# Ejercicio 39: Traducción de a*b + c*d a: infija, postfija, código 3-dir, cuádruplas

print("=" * 60)
print("Expresión: a * b + c * d")
print("=" * 60)

print("\n1. NOTACIÓN INFIJA (original):")
print("   a * b + c * d")

print("\n2. NOTACIÓN POSTFIJA (RPN):")
postfija = infija_a_postfija("a * b + c * d")
print(f"   {postfija}")

print("\n3. CÓDIGO DE TRES DIRECCIONES:")
c3d = [
    "t1 = a * b",
    "t2 = c * d",
    "t3 = t1 + t2",
]
for i, inst in enumerate(c3d, 1):
    print(f"   ({i}) {inst}")

print("\n4. CUÁDRUPLAS:")
cuads = [
    ("*", "a", "b", "t1"),
    ("*", "c", "d", "t2"),
    ("+", "t1", "t2", "t3"),
]
print(f"   {'Op':^6} {'Arg1':^5} {'Arg2':^5} {'Res':^5}")
for op, a1, a2, r in cuads:
    print(f"   {op:^6} {a1:^5} {a2:^5} {r:^5}")

print("\n5. TRIPLETAS:")
trips = [
    (0, "*", "a", "b"),
    (1, "*", "c", "d"),
    (2, "+", "(0)", "(1)"),
]
print(f"   {'Pos':>3} {'Op':^6} {'Arg1':^6} {'Arg2':^6}")
for pos, op, a1, a2 in trips:
    print(f"   ({pos}) {op:^6} {a1:^6} {a2:^6}")

print("\n6. CÓDIGO DE PILA:")
codigo_pila = generar_codigo_pila(postfija)
for i, inst in enumerate(codigo_pila, 1):
    print(f"   {i:>2}. {inst}")

### ✏️ Ejercicio 40: Resumen visual interactivo — pipeline de compilación

In [ ]:
# Ejercicio 40: Pipeline completo — de código fuente a código intermedio

import textwrap

def caja(titulo, contenido, ancho=60):
    borde = "─" * (ancho - 2)
    print(f"┌{borde}┐")
    titulo_centrado = titulo.center(ancho - 2)
    print(f"│{titulo_centrado}│")
    print(f"├{borde}┤")
    for linea in contenido:
        linea_ajustada = linea.ljust(ancho - 2)
        print(f"│{linea_ajustada}│")
    print(f"└{borde}┘")
    print("         │")
    print("         ▼")

# Programa fuente
print("\n📋 PIPELINE DE COMPILACIÓN — Ejemplo completo\n")
print("Programa fuente:")
print("  z = (x + y) * 2")
print()

caja("1. ANÁLISIS LÉXICO (Tokens)", [
    "  ID(z)  OP(=)  LPAREN  ID(x)  OP(+)  ID(y)  RPAREN",
    "  OP(*)  NUM(2)",
])

caja("2. ANÁLISIS SINTÁCTICO (AST)", [
    "       ASIG(=)",
    "       /      \\",
    "      z       MULT",
    "             /    \\",
    "           SUMA    2",
    "           /  \\",
    "          x    y",
])

caja("3. ANÁLISIS SEMÁNTICO (tipos)", [
    "  x: int, y: int, z: int",
    "  SUMA(int, int) → int  ✅",
    "  MULT(int, int) → int  ✅",
])

caja("4. GENERACIÓN CÓDIGO INTERMEDIO (3-dir)", [
    "  (1) t1 = x + y",
    "  (2) t2 = t1 * 2",
    "  (3) z  = t2",
])

caja("5. OPTIMIZACIÓN DE CÓDIGO INTERMEDIO", [
    "  (1) t1 = x + y",
    "  (2) t2 = t1 + t1   ← reducción fuerza: *2 → +",
    "  (3) z  = t2",
])

print("┌──────────────────────────────────────────────────────────┐")
print("│               6. GENERACIÓN CÓDIGO OBJETO                │")
print("├──────────────────────────────────────────────────────────┤")
print("│  LOAD  R1, x    ; R1 = x                                 │")
print("│  LOAD  R2, y    ; R2 = y                                 │")
print("│  ADD   R1, R2   ; R1 = x + y                            │")
print("│  ADD   R1, R1   ; R1 = (x+y)*2                          │")
print("│  STORE z, R1    ; z = R1                                 │")
print("└──────────────────────────────────────────────────────────┘")
print()
print("🎉 ¡Pipeline de compilación completo!")
print("   La Unidad 3 cubre la etapa 4 (y partes de 5).")

---
## 📚 Resumen de Conceptos Clave

| Concepto | Descripción | Ejemplo |
|---|---|---|
| **Código 3 dir.** | Máx. 3 operandos por instrucción | `t1 = a + b` |
| **Cuádrupla** | `(op, arg1, arg2, resultado)` | `(*, a, b, t1)` |
| **Tripleta** | `(op, arg1, arg2)`, resultado por posición | `(0: *, a, b)` |
| **RPN / Postfija** | Operador después de operandos | `a b +` |
| **Temporal** | Variable auxiliar generada por el compilador | `t1, t2, ...` |
| **Etiqueta** | Marca de posición para saltos | `L1:, L2:` |
| **Bloque básico** | Secuencia sin saltos internos | — |
| **CFG** | Grafo de flujo de control entre bloques | — |
| **CSE** | Eliminación de subexpresiones comunes | — |
| **Reducción de fuerza** | Op. costosa → Op. barata | `x*2 → x+x` |

---
### 🔗 Recursos de estudio adicionales
- **Libro**: Aho, Lam, Sethi, Ullman — *Compilers: Principles, Techniques, and Tools* (Dragon Book)
- **Capítulo relevante**: Capítulo 6 — Intermediate Code Generation
- **Práctica**: Modificar los ejercicios para agregar nuevos operadores o estructuras de control
